In [ ]:
# nama file: jalankan_prediksi_firebase_rata_rata.py

import time
import firebase_admin
from firebase_admin import credentials, db
import joblib
import numpy as np
import pandas as pd
from collections import deque # ✅ Kita gunakan deque untuk menyimpan histori prediksi

# --- KONFIGURASI ---
SERVICE_ACCOUNT_PATH = "/content/drive/MyDrive/Colab Notebooks/tubes-iot-1bb8c-firebase-adminsdk-fbsvc-245e01333e.json"
DATABASE_URL = "https://tubes-iot-1bb8c-default-rtdb.firebaseio.com/"
NAMA_FILE_MODEL = "/content/drive/MyDrive/Colab Notebooks/glucose_model_v3.pkl"

# --- Konfigurasi untuk Prediksi Rata-rata ---
JUMLAH_SAMPEL_UNTUK_RATA_RATA = 10 # Ambil rata-rata dari 10 prediksi terakhir
histori_prediksi = deque(maxlen=JUMLAH_SAMPEL_UNTUK_RATA_RATA)

# --- Inisialisasi ---
try:
    if not firebase_admin._apps:
        cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
        firebase_admin.initialize_app(cred, {'databaseURL': DATABASE_URL})
    print("✅ Koneksi Firebase berhasil.")

    model = joblib.load(NAMA_FILE_MODEL)
    print(f"✅ Model '{NAMA_FILE_MODEL}' berhasil dimuat.")

except FileNotFoundError:
    print(f"❌ File '{NAMA_FILE_MODEL}' atau '{SERVICE_ACCOUNT_PATH}' tidak ditemukan.")
    exit()
except Exception as e:
    print(f"❌ Terjadi error saat inisialisasi: {e}")
    exit()


def predict_glucose(ir_val, red_val, bpm_val):
    try:
        input_df = pd.DataFrame([{'IR': ir_val, 'RED': red_val, 'BPM': bpm_val}])
        prediction = model.predict(input_df)
        return prediction[0] # Kembalikan nilai float mentah
    except Exception as e:
        print(f"❌ Error saat prediksi: {e}")
        return -1


# --- Loop Utama ---
print(f"\n🚀 Memulai loop... Akan menghitung rata-rata dari {JUMLAH_SAMPEL_UNTUK_RATA_RATA} sampel.")
last_input_data = None

while True:
    try:
        input_ref = db.reference("/glucose_predict/input")
        data = input_ref.get()

        # Hanya proses jika ada data BARU yang masuk
        if data and data != last_input_data:
            last_input_data = data

            ir = data.get('ir', 0)
            red = data.get('red', 0)
            bpm = data.get('bpm', 0)

            print(f"\n📨 Data baru diterima -> [IR: {ir}, RED: {red}, BPM: {bpm}]")

            # Lakukan prediksi tunggal
            prediksi_tunggal = predict_glucose(ir, red, bpm)

            if prediksi_tunggal != -1:
                # Tambahkan prediksi ke histori
                histori_prediksi.append(prediksi_tunggal)
                print(f"   -> Prediksi tunggal: {prediksi_tunggal:.1f}. Histori: {len(histori_prediksi)}/{JUMLAH_SAMPEL_UNTUK_RATA_RATA}")

            # Jika histori sudah cukup penuh, hitung rata-rata dan kirim
            if len(histori_prediksi) == JUMLAH_SAMPEL_UNTUK_RATA_RATA:
                rata_rata_prediksi = int(round(np.mean(list(histori_prediksi))))

                print(f"   ✅ Histori penuh. Menghitung rata-rata...")
                print(f"   ✅ Prediksi Final (Rata-rata) -> {rata_rata_prediksi} mg/dL")

                # Kirim hasil akhir ke Firebase
                pred_ref = db.reference("/glucose_predict/prediction")
                pred_ref.set(rata_rata_prediksi)
                print("   📤 Hasil prediksi final berhasil dikirim ke Firebase.")

                # Kosongkan histori agar pengukuran baru dimulai
                histori_prediksi.clear()

    except Exception as e:
        print(f"❌ Error dalam loop utama: {e}")
        time.sleep(10)

    time.sleep(3) # Cek data baru setiap 3 detik